# Step 1 — baseline FVE (Kaggle, 2×T4, fp16 + fp32 check)
Settings: **Accelerator GPU T4 ×2**, **Internet on**, phone-verified account.
Add-ons → Secrets: `HF_TOKEN`, `WANDB_API_KEY` (attach both to this notebook).
Resumable: re-run all cells after a disconnect; finished shards are pulled from the private HF repo.

In [ ]:
# Weights (~45 GB) go on the big ephemeral root disk, not /kaggle/working (20 GB).
import os
os.environ["HF_HOME"] = "/tmp/hf"
!df -h /tmp /kaggle/working | cat
!nvidia-smi --query-gpu=name,memory.total --format=csv
import shutil
assert shutil.which("nvidia-smi"), "NO GPU: turn on the GPU accelerator (see the first cell) and run again"

In [ ]:
import os, subprocess
REPO = "https://github.com/trinath122/open-nla-audits"
COMMIT = "main"  # pin to a commit hash for the real run
# Absolute path + always fetch: a restarted session must not run stale code.
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
SRC = os.path.join(WORK, "open-nla-audits")
if not os.path.exists(SRC):
    subprocess.run(["git", "clone", REPO, SRC], check=True)
os.chdir(SRC)
subprocess.run(["git", "fetch", "origin"], check=True)
target = COMMIT if COMMIT != "main" else "origin/main"
subprocess.run(["git", "checkout", "--detach", target], check=True)
print(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
# keep the platform's CUDA torch; install the rest
# Platforms ship newer Pythons (Kaggle: 3.13); the code is tested on 3.11 but has no 3.11-only deps.
!pip -q install -e . --no-deps --ignore-requires-python
import sys
sys.path.insert(0, os.path.abspath("src"))  # the running kernel won't see a fresh editable install
import nla_audit; print("nla_audit from", nla_audit.__file__)
!pip -q install "transformers>=5,<6" "accelerate>=1.0" "datasets>=2.20" "huggingface-hub>=0.24" safetensors wandb python-dotenv pyarrow pyyaml

In [ ]:
# Secrets -> environment so the `!python` subprocess inherits them. Values are never printed.
from nla_audit.runtime.env import load_secrets
found = load_secrets(("HF_TOKEN", "WANDB_API_KEY"))
print("secrets found:", found)
assert "HF_TOKEN" in found, "add HF_TOKEN to this platform's secrets"
os.environ.setdefault("WANDB_PROJECT", "open-nla-audits")

In [ ]:
PROFILE = "kaggle"
!python -m nla_audit.experiments.step1_baseline run --config configs/step1.yaml --profile {PROFILE} --sync

In [ ]:
import json
path = f"runs/step1_baseline/{PROFILE}/summary.json"
assert os.path.exists(path), "the run cell above failed - read its error (scroll up), this cell only shows results"
s = json.load(open(path))
for tag, rep in s["tags"].items():
    print(tag, json.dumps(rep["verdict"], indent=1))
    for k in ("overall", "ufw", "wildchat"):
        r = rep[k]
        print(f"  {k:9s} FVE(own) {r['fve_own']['fve']:.3f} [{r['fve_own']['lo']:.3f}, {r['fve_own']['hi']:.3f}]"
              f"  FVE(0.7335) {r['fve_reference']['fve']:.3f}  parse-fail {r['parse_fail_rate']:.2%}  CJK {r['cjk_any_rate']:.2%}")
    print("  shuffled control:", rep["shuffled_control"])
if "precision_check" in s:
    print("precision check:", json.dumps(s["precision_check"], indent=1))
print("L20 mean norm:", s["mean_norm"]["mean_norm"])
print("monitors:", {k: (v["max_abs"], v["any_nonfinite"]) for k, v in s.get("monitors", {}).items()})